In [ ]:
from dataclasses import dataclass, asdict

import numpy as np
import matplotlib.pyplot as plt
import qnm


# Physical constants
G = 6.67430e-11
C = 299_792_458.0
MSUN = 1.988409870698051e30

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

@dataclass
class KerrRemnant:
    mass_msun: float
    chi: float

    def __post_init__(self):
        if self.mass_msun <= 0:
            raise ValueError("mass_msun must be positive")
        if not (-1.0 < self.chi < 1.0):
            raise ValueError("chi must satisfy -1 < chi < 1")

    @property
    def mass_si(self) -> float:
        return self.mass_msun * MSUN

    @property
    def time_unit(self) -> float:
        return G * self.mass_si / C**3

def get_kerr_qnm(remnant: KerrRemnant, l: int, m: int, n: int, s: int = -2) -> dict:
    mode_seq = qnm.modes_cache(s=s, l=l, m=m, n=n)
    result = mode_seq(a=remnant.chi)
    omega = result[0]

    omega_r_geom = float(np.real(omega))
    omega_i_geom = float(-np.imag(omega))

    M_sec = remnant.time_unit
    omega_r_si = omega_r_geom / M_sec
    omega_i_si = omega_i_geom / M_sec

    return {
        "l": l,
        "m": m,
        "n": n,
        "omega_complex_geom": omega,
        "omega_r_geom": omega_r_geom,
        "omega_i_geom": omega_i_geom,
        "omega_r_si": omega_r_si,
        "omega_i_si": omega_i_si,
        "f_hz": omega_r_si / (2.0 * np.pi),
        "tau_s": 1.0 / omega_i_si,
    }

@dataclass
class RingdownMode:
    l: int
    m: int
    n: int
    amplitude: float
    phase: float


@dataclass
class RingdownSignal:
    t: np.ndarray
    h_complex: np.ndarray

    @property
    def h_plus(self) -> np.ndarray:
        return np.real(self.h_complex)

    @property
    def h_cross(self) -> np.ndarray:
        return -np.imag(self.h_complex)

def component_strain_from_freqs(
    amplitude: float,
    phase: float,
    omega_r_si: float,
    omega_i_si: float,
    t: np.ndarray,
    t0: float = 0.0,
) -> np.ndarray:
    dt = t - t0
    mask = dt >= 0.0

    h = np.zeros_like(t, dtype=np.complex128)
    h[mask] = (
        amplitude
        * np.exp(-omega_i_si * dt[mask])
        * np.exp(-1j * (omega_r_si * dt[mask] + phase))
    )
    return h


def generate_gr_ringdown(
    remnant: KerrRemnant,
    modes: list[RingdownMode],
    duration: float,
    sample_rate: float,
    t0: float = 0.0,
    s: int = -2,
) -> RingdownSignal:
    t = np.arange(0.0, duration, 1.0 / sample_rate)
    h = np.zeros_like(t, dtype=np.complex128)

    for mode in modes:
        q = get_kerr_qnm(remnant, mode.l, mode.m, mode.n, s=s)
        h += component_strain_from_freqs(
            amplitude=mode.amplitude,
            phase=mode.phase,
            omega_r_si=q["omega_r_si"],
            omega_i_si=q["omega_i_si"],
            t=t,
            t0=t0,
        )

    return RingdownSignal(t=t, h_complex=h)

@dataclass
class ModeDeviation:
    d_omega_frac: float = 0.0
    d_tau_frac: float = 0.0
    d_amp_frac: float = 0.0
    d_phase: float = 0.0

def mode_key(l: int, m: int, n: int) -> tuple[int, int, int]:
    return (l, m, n)

def get_mode_deviation(
    deviations: dict[tuple[int, int, int], ModeDeviation],
    l: int,
    m: int,
    n: int,
) -> ModeDeviation:
    return deviations.get((l, m, n), ModeDeviation())

def deform_qnm_physical_parameters(
    omega_r_si: float,
    omega_i_si: float,
    deviation: ModeDeviation,
) -> tuple[float, float]:
    if 1.0 + deviation.d_tau_frac <= 0.0:
        raise ValueError("d_tau_frac must satisfy 1 + d_tau_frac > 0")

    omega_r_new = omega_r_si * (1.0 + deviation.d_omega_frac)
    omega_i_new = omega_i_si / (1.0 + deviation.d_tau_frac)

    return omega_r_new, omega_i_new

def generate_modified_ringdown(
    remnant: KerrRemnant,
    modes: list[RingdownMode],
    deviations: dict[tuple[int, int, int], ModeDeviation],
    duration: float,
    sample_rate: float,
    t0: float = 0.0,
    s: int = -2,
) -> RingdownSignal:
    t = np.arange(0.0, duration, 1.0 / sample_rate)
    h = np.zeros_like(t, dtype=np.complex128)

    for mode in modes:
        q = get_kerr_qnm(remnant, mode.l, mode.m, mode.n, s=s)
        dev = get_mode_deviation(deviations, mode.l, mode.m, mode.n)

        omega_r_new, omega_i_new = deform_qnm_physical_parameters(
            omega_r_si=q["omega_r_si"],
            omega_i_si=q["omega_i_si"],
            deviation=dev,
        )

        amp_new = mode.amplitude * (1.0 + dev.d_amp_frac)
        phase_new = mode.phase + dev.d_phase

        h += component_strain_from_freqs(
            amplitude=amp_new,
            phase=phase_new,
            omega_r_si=omega_r_new,
            omega_i_si=omega_i_new,
            t=t,
            t0=t0,
        )

    return RingdownSignal(t=t, h_complex=h)

remnant = KerrRemnant(mass_msun=60.0, chi=0.7)

modes = [
    RingdownMode(l=2, m=2, n=0, amplitude=1.00, phase=0.0),
    RingdownMode(l=2, m=2, n=1, amplitude=0.45, phase=0.3),
    RingdownMode(l=3, m=3, n=0, amplitude=0.25, phase=-0.5),
]

duration = 0.05
sample_rate = 8192
t0 = 0.002

signal_gr = generate_gr_ringdown(
    remnant=remnant,
    modes=modes,
    duration=duration,
    sample_rate=sample_rate,
    t0=t0,
)

deviations = {
    (2, 2, 0): ModeDeviation(d_omega_frac=0.02, d_tau_frac=0.10, d_amp_frac=0.00, d_phase=0.00),
    (2, 2, 1): ModeDeviation(d_omega_frac=-0.01, d_tau_frac=0.15, d_amp_frac=0.05, d_phase=0.10),
    (3, 3, 0): ModeDeviation(d_omega_frac=0.03, d_tau_frac=-0.05, d_amp_frac=-0.10, d_phase=-0.08),
}

signal_mg = generate_modified_ringdown(
    remnant=remnant,
    modes=modes,
    deviations=deviations,
    duration=duration,
    sample_rate=sample_rate,
    t0=t0,
)

plt.figure()
plt.plot(signal_gr.t, signal_gr.h_plus, label="GR")
plt.plot(signal_mg.t, signal_mg.h_plus, label="Modified gravity", alpha=0.9)
plt.xlabel("Time [s]")
plt.ylabel(r"$h_+$ [arb.]")
plt.title("GR vs mode-dependent modified-gravity ringdown")
plt.legend()
plt.show()

plt.figure()
plt.plot(signal_gr.t, np.abs(signal_gr.h_complex), "--", label="GR envelope")
plt.plot(signal_mg.t, np.abs(signal_mg.h_complex), "--", label="MG envelope")
plt.xlabel("Time [s]")
plt.ylabel("Envelope [arb.]")
plt.title("Envelope comparison")
plt.legend()
plt.show()

plt.figure()

for mode in modes:
    sig_gr_mode = generate_gr_ringdown(
        remnant=remnant,
        modes=[mode],
        duration=duration,
        sample_rate=sample_rate,
        t0=t0,
    )
    sig_mg_mode = generate_modified_ringdown(
        remnant=remnant,
        modes=[mode],
        deviations=deviations,
        duration=duration,
        sample_rate=sample_rate,
        t0=t0,
    )

    label = f"({mode.l},{mode.m},{mode.n})"
    plt.plot(sig_gr_mode.t, sig_gr_mode.h_plus, label=f"{label} GR")
    plt.plot(sig_mg_mode.t, sig_mg_mode.h_plus, "--", label=f"{label} MG")

plt.xlabel("Time [s]")
plt.ylabel(r"$h_+$ [arb.]")
plt.title("Individual mode deformations")
plt.legend(ncol=2, fontsize=9)
plt.show()

summary_table = []

for mode in modes:
    q = get_kerr_qnm(remnant, mode.l, mode.m, mode.n)
    dev = get_mode_deviation(deviations, mode.l, mode.m, mode.n)

    omega_r_new, omega_i_new = deform_qnm_physical_parameters(
        q["omega_r_si"],
        q["omega_i_si"],
        dev,
    )

    summary_table.append({
        "mode": f"({mode.l},{mode.m},{mode.n})",
        "f_GR_Hz": q["f_hz"],
        "tau_GR_ms": 1e3 * q["tau_s"],
        "d_omega_frac": dev.d_omega_frac,
        "d_tau_frac": dev.d_tau_frac,
        "d_amp_frac": dev.d_amp_frac,
        "d_phase": dev.d_phase,
        "f_MG_Hz": omega_r_new / (2.0 * np.pi),
        "tau_MG_ms": 1e3 / omega_i_new,
    })

summary_table

def waveform_inner_product(h1: np.ndarray, h2: np.ndarray, dt: float) -> float:
    return float(np.sum(h1 * h2) * dt)

def waveform_norm(h: np.ndarray, dt: float) -> float:
    return np.sqrt(max(waveform_inner_product(h, h, dt), 0.0))

def waveform_overlap(h1: np.ndarray, h2: np.ndarray, dt: float) -> float:
    n1 = waveform_norm(h1, dt)
    n2 = waveform_norm(h2, dt)
    if n1 == 0.0 or n2 == 0.0:
        return 0.0
    return waveform_inner_product(h1, h2, dt) / (n1 * n2)

def waveform_mismatch(h1: np.ndarray, h2: np.ndarray, dt: float) -> float:
    return 1.0 - waveform_overlap(h1, h2, dt)

dt = signal_gr.t[1] - signal_gr.t[0]

ov = waveform_overlap(signal_gr.h_plus, signal_mg.h_plus, dt)
mm = waveform_mismatch(signal_gr.h_plus, signal_mg.h_plus, dt)

print(f"Overlap(GR, MG)   = {ov:.6f}")
print(f"Mismatch(GR, MG)  = {mm:.6f}")

domega_grid = np.linspace(-0.05, 0.05, 81)
mismatch_values = []

for x in domega_grid:
    trial_deviations = {
        (2, 2, 0): ModeDeviation(d_omega_frac=float(x), d_tau_frac=0.0, d_amp_frac=0.0, d_phase=0.0)
    }

    sig_trial = generate_modified_ringdown(
        remnant=remnant,
        modes=modes,
        deviations=trial_deviations,
        duration=duration,
        sample_rate=sample_rate,
        t0=t0,
    )

    mismatch_values.append(
        waveform_mismatch(signal_gr.h_plus, sig_trial.h_plus, dt)
    )

mismatch_values = np.array(mismatch_values)

plt.figure()
plt.plot(domega_grid, mismatch_values)
plt.xlabel(r"$\delta\omega_{220}$")
plt.ylabel("Mismatch")
plt.title(r"Mismatch vs frequency deformation of the $(2,2,0)$ mode")
plt.show()

domega_vals = np.linspace(-0.04, 0.04, 41)
dtau_vals = np.linspace(-0.20, 0.20, 41)

mismatch_grid = np.zeros((len(dtau_vals), len(domega_vals)))

for i, dtau in enumerate(dtau_vals):
    for j, domega in enumerate(domega_vals):
        trial_deviations = {
            (2, 2, 0): ModeDeviation(
                d_omega_frac=float(domega),
                d_tau_frac=float(dtau),
                d_amp_frac=0.0,
                d_phase=0.0,
            )
        }

        sig_trial = generate_modified_ringdown(
            remnant=remnant,
            modes=modes,
            deviations=trial_deviations,
            duration=duration,
            sample_rate=sample_rate,
            t0=t0,
        )

        mismatch_grid[i, j] = waveform_mismatch(signal_gr.h_plus, sig_trial.h_plus, dt)

plt.figure()
plt.imshow(
    mismatch_grid,
    origin="lower",
    aspect="auto",
    extent=[domega_vals[0], domega_vals[-1], dtau_vals[0], dtau_vals[-1]],
)
plt.colorbar(label="Mismatch")
plt.xlabel(r"$\delta\omega_{220}$")
plt.ylabel(r"$\delta\tau_{220}$")
plt.title(r"Mismatch map for dominant-mode deformation")
plt.show()